# Data types and functions: CAST, text, numbers and dates

Raw data often arrives as text. CAST turns a value into the right type, and try_cast and try_to_date return NULL instead of failing when a value is bad. Built-in functions clean text (trim, lower, initcap, concat_ws), round and divide numbers, and work with dates (date_add, datediff, trunc, date_format).

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Data types and functions: CAST, text, numbers and dates](https://www.datapreplab.com/hub#topic/sql-functions-types) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** convert a raw, all-text extract into clean, typed columns and predict each result first.

**Part A: raw data, everything as text.**

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW raw_customers AS
SELECT * FROM VALUES
  ('1', '  asha KHAN ', '120.50', '2026-01-15'),
  ('2', 'ben ode',      '75',     '2026-02-03'),
  ('3', 'CHEN li',      'n/a',    '2026-02-30')
AS t(id, name, amount, signup);

In [ ]:
%sql
DESCRIBE raw_customers;

**Part B: clean and convert safely.** Predict which values become NULL, then run:

In [ ]:
%sql
SELECT CAST(id AS INT)                   AS id,
       initcap(trim(name))               AS name,
       try_cast(amount AS DECIMAL(10,2)) AS amount,
       try_to_date(signup)               AS signup
FROM raw_customers
ORDER BY id;

**Part C: find what failed.** Rows where the raw value was present but the converted value is NULL need attention:

In [ ]:
%sql
SELECT id, amount, signup
FROM raw_customers
WHERE try_cast(amount AS DECIMAL(10,2)) IS NULL
   OR try_to_date(signup) IS NULL;

**Part D: numbers and dates.**

In [ ]:
%sql
SELECT 5 / 2                                          AS divide,
       5 div 2                                        AS whole,
       round(10 / 3, 2)                               AS rounded,
       datediff(DATE'2026-03-01', DATE'2026-02-01')   AS days,
       trunc(DATE'2026-02-17', 'MM')                  AS month_start,
       date_format(DATE'2026-02-17', 'yyyy-MM')       AS month_label;

**Part E: the strict conversion fails.** Now use CAST instead of try_cast. Predict which value breaks it, then run:

In [ ]:
%sql
SELECT CAST(amount AS DECIMAL(10,2)) AS amount FROM raw_customers;

The query stops with an error about `'n/a'`. That is ANSI mode protecting you from silently wrong data.

Expected evidence:

```text
Part A -> 3 columns, all string
Part B -> 1 Asha Khan 120.50 2026-01-15 ; 2 Ben Ode 75.00 2026-02-03 ; 3 Chen Li NULL NULL
Part C -> one row: 3 n/a 2026-02-30
Part D -> 2.5, 2, 3.33, 28, 2026-02-01, 2026-02
Part E -> error: 'n/a' cannot be cast to DECIMAL(10,2)
```

Part E is meant to fail: it is the last step so that **Run all** still runs everything before it.